# CodeLlama Automatic Evaluation

## Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import nltk
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


## Model

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hf_token = os.getenv("HF_TOKEN")

model_name = "codellama/CodeLlama-7b-Instruct-hf"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token
)


## Source Files

In [ ]:

def get_all_code_files(repo_path):
    code_files = []
    extensions = (".py", ".java", ".kt", ".js")

    for root, _, files in os.walk(repo_path):
        for file_name in files:
            if not file_name.endswith(extensions):
                continue

            full_path = os.path.abspath(os.path.join(root, file_name))

            try:
                with open(full_path, "r", encoding="utf-8", errors="ignore") as file:
                    content = file.read()
            except OSError:
                continue

            if len(content) <= 50:
                continue

            code_files.append({
                "file_name": os.path.relpath(full_path, repo_path),
                "full_path": full_path,
                "content": content[:1000],
            })

    return code_files


## Prompting Strategies

In [ ]:

def prompt_zero(code, readme, complexity, comments, commits):
    return f"""
You are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for the following assessment aspects: Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.

1. Semantic Correctness: The summary must accurately represent the code's logic, control flow, and technical behavior without misinterpretation.
2. Informativeness: The summary must include key details such as the primary objective, major input/output, and relevant dependencies.
3. Readability: The summary must use straightforward, structured language that is easy for other developers to comprehend.
4. Usefulness: The summary must provide context that aids in code maintenance and program understanding.
5. Overall Quality: The summary must be professional, concise, and adhere to industry-standard documentation practices.

Inputs:
- Code: {code}
- README: {readme}
- Complexity: {complexity}
- Comments: {comments}
- Commits: {commits}
- Code Size: {len(code)}

Output ONLY the final summary.
"""


def prompt_few(code, readme, complexity, comments, commits):
    return f"""
You are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for the following assessment aspects: Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.

1. Semantic Correctness: The summary must accurately represent the code's logic, control flow, and technical behavior without misinterpretation.
2. Informativeness: The summary must include key details such as the primary objective, major input/output, and relevant dependencies.
3. Readability: The summary must use straightforward, structured language that is easy for other developers to comprehend.
4. Usefulness: The summary must provide context that aids in code maintenance and program understanding.
5. Overall Quality: The summary must be professional, concise, and adhere to industry-standard documentation practices.

Example:

Inputs:
- Code:
def load_config(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError("Config missing")
    with open(filepath, "r") as f:
        return json.load(f)
- README: Utility to initialize system settings.
- Complexity: Cyclomatic Complexity: 2
- Comments: Checks for file existence before loading JSON.
- Commits: Added error handling for missing configuration files.
- Code Size: 5 lines

Output:
The `load_config` function initializes system settings by reading and parsing a JSON configuration file. It validates file existence before loading the data and raises `FileNotFoundError` when the configuration file is unavailable.

Actual task:

Inputs:
- Code: {code}
- README: {readme}
- Complexity: {complexity}
- Comments: {comments}
- Commits: {commits}
- Code Size: {len(code)}

Output:
"""


def prompt_adv(code, readme, comments, commits, complexity):
    return f"""
You are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.

Follow this reasoning process:
1. Identify the code's main purpose, inputs, and outputs.
2. Analyze key operations and control flow.
3. Incorporate the available context: Complexity: {complexity}, Comments: {comments}, Commits: {commits}, README: {readme}, Code Size: {len(code)}.
4. Remove redundancy and use precise technical language.
5. Produce a cohesive 2-3 sentence summary focused on functionality and purpose.

Output ONLY the final summary. Do not include reasoning steps, bullet points, or explanations.

Code:
{code}

Summary:
"""


## Evaluation

In [ ]:

smooth = SmoothingFunction().method1
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def generate_summary(prompt_text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert software engineer. Provide a concise, clear, "
                "and direct summary of the functionality of the provided code. "
                "Do not repeat the code."
            ),
        },
        {"role": "user", "content": prompt_text},
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0][input_length:]
    result = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return result.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t").strip()


def calculate_metrics(prediction, reference):
    if not prediction or prediction.isspace():
        return 0.0, 0.0, 0.0

    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    try:
        bleu = sentence_bleu(
            [ref_tokens],
            pred_tokens,
            smoothing_function=smooth,
        )
        rouge = scorer.score(reference, prediction)["rougeL"].fmeasure
        meteor = meteor_score([ref_tokens], pred_tokens)
    except (ValueError, TypeError, LookupError):
        return 0.0, 0.0, 0.0

    return round(bleu, 4), round(rouge, 4), round(meteor, 4)


## Execution

In [ ]:

repo_github_links = {
    "storm": "https://github.com/apache/storm",
    "butterknife": "https://github.com/JakeWharton/butterknife",
    "crate": "https://github.com/crate/crate",
    "hystrix": "https://github.com/Netflix/Hystrix",
    "jabref": "https://github.com/JabRef/jabref",
    "jcabi": "https://github.com/jcabi/jcabi-github",
    "openmicroscopy": "https://github.com/ome/openmicroscopy",
    "presto": "https://github.com/prestodb/presto",
    "rxandroid": "https://github.com/ReactiveX/RxAndroid",
    "sponge": "https://github.com/SpongePowered/SpongeAPI",
    "springboot": "https://github.com/spring-projects/spring-boot",
    "okhttp": "https://github.com/square/okhttp",
    "retrofit": "https://github.com/square/retrofit",
    "wordpress": "https://github.com/wordpress-mobile/WordPress-Android",
}

repositories = [
    "butterknife",
    "crate",
    "hystrix",
    "jabref",
    "jcabi",
    "okhttp",
    "openmicroscopy",
    "presto",
    "retrofit",
    "rxandroid",
    "sponge",
    "storm",
    "springboot",
    "wordpress",
]



results = []
reference_text = "This software module implements core functionality and logic."

readme = "This repository contains backend software components."
complexity = "O(N)"
comments = "Implementation of core algorithms."
commits = "Initial codebase commit."

for repo_name in tqdm(repositories, desc="Repositories"):
    repo_path = f"repos/{repo_name}"
    files_data = get_all_code_files(repo_path)
    file_target = files_data[:1000]
    base_github_url = repo_github_links.get(repo_name, "")

    for file_info in tqdm(file_target, desc=repo_name, leave=False):
        file_name = file_info["file_name"]
        code_content = file_info["content"]

        try:
            if base_github_url:
                safe_path = file_name.replace("\\", "/")
                github_url = f"{base_github_url}/blob/master/{safe_path}"
                file_link = f'=HYPERLINK("{github_url}")'
            else:
                file_link = f'=HYPERLINK("{file_info["full_path"]}")'

            zero_prompt = prompt_zero(
                code_content, readme, complexity, comments, commits
            )
            few_prompt = prompt_few(
                code_content, readme, complexity, comments, commits
            )
            adv_prompt = prompt_adv(
                code_content, readme, comments, commits, complexity
            )

            zero_summary = generate_summary(zero_prompt)
            few_summary = generate_summary(few_prompt)
            adv_summary = generate_summary(adv_prompt)

            bz, rz, mz = calculate_metrics(zero_summary, reference_text)
            bf, rf, mf = calculate_metrics(few_summary, reference_text)
            ba, ra, ma = calculate_metrics(adv_summary, reference_text)

            results.append({
                "Repository": repo_name,
                "File Name": file_name,
                "File Link": file_link,
                "Zero Summary": zero_summary,
                "BLEU Zero": bz,
                "ROUGE Zero": rz,
                "METEOR Zero": mz,
                "Few Summary": few_summary,
                "BLEU Few": bf,
                "ROUGE Few": rf,
                "METEOR Few": mf,
                "Adv Summary": adv_summary,
                "BLEU Adv": ba,
                "ROUGE Adv": ra,
                "METEOR Adv": ma,
            })

        except Exception as exc:
            tqdm.write(f"Skipped {file_name}: {exc}")

results_df = pd.DataFrame(results)
results_df.to_excel("CodeLlama_Automatic_Evaluation_Results.xlsx", index=False)
display(results_df)
